# Compare our fine-tuned checkpoint against the released public one  ·  Person B

Two separate comparisons, both read-only — nothing here trains or modifies either
checkpoint:

**A. HebNLI test-set accuracy.** `03_eval_nli.ipynb` measured `alephbert-hebnli-clean`
(ours) at accuracy 0.7961 / macro-F1 0.7940 on the 883-row clean test set. This
notebook measures the released checkpoint (`oriel9p/AlephBERT-FT-HebNLI-LCHAIM`,
what `nli_rerank.py` defaulted to before) on the *same* file, for a labelled
reference point.

**This is not a fair comparison, and the notebook says so at the point it matters:**
the released checkpoint was fine-tuned on *all* of HebNLI rather than a train/test
split, so this test set was very likely part of its own training data. A strong
score from it measures memorisation, not generalisation — `eval_nli.py` stamps a
`caveat` into its summary file for exactly this reason.

**B. The actual negation-probe experiment.** `nli_rerank.py` blends embedding cosine
with an NLI judgement; until now it always used the released checkpoint by default,
with no way to point the harness at ours. This notebook runs `src.harness.run_eval`
with `--interventions nli_rerank` once per checkpoint and puts both rows in the same
results table (`results/results_nli_rerank.csv`) — this is the one that answers
"does our decontaminated model actually help the probe measurement."

Runs in two places (Colab web UI, VS Code + Colab extension), same as the other
notebooks — the checkpoint lives on Drive and both embedder downloads and the
released checkpoint need a GPU-having session. Run cell by cell.

## Where everything ends up

| what | where | survives a reset? |
|---|---|---|
| cloned repo | VM disk | no |
| `data/raw/hebnli_test_clean.jsonl` | VM disk, regenerated from source | no |
| our checkpoint | **Drive**, read-only | yes |
| the released (LCHAIM) checkpoint | downloaded from the HF hub, cached on VM disk | no |
| `results/nli_test_alephbert-hebnli-lchaim.json` + predictions csv | VM disk, then downloaded | via git |
| `results/results_nli_rerank.csv` | VM disk, then downloaded | via git |

Nothing here writes to `checkpoints/` on Drive. The released checkpoint's results get
their own pair of files, named the same way ours are
(`nli_test_alephbert-hebnli-clean.json` / `..._lchaim.json`) rather than sharing a file
with ours or living under a generic name.

## 1. Setup

**Prints** &nbsp; Nothing. This cell only defines a helper.

**Writes** &nbsp; Nothing.

In [4]:
# Secrets three ways: Colab's store, then the environment, then a prompt. The VS Code
# extension cannot reach Colab's secret store, so the fallbacks are what make this
# notebook portable. getpass also keeps the token out of the saved output.
import os, subprocess, getpass

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f'{name}: from Colab secrets')
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        print(f'{name}: from environment')
        return value.strip()
    return getpass.getpass(f'{name}: ').strip()

**Prints** &nbsp; Python and torch versions, the GPU name, and which `google.colab`
modules import. `cuda: True` plus a real GPU are worth confirming before the released
checkpoint's own download, which is a few hundred MB.

**Writes** &nbsp; Nothing.

In [5]:
# What are we running on? Answers 'will this work here' before anything slow.
import platform
print('python      ', platform.python_version())
print('cwd         ', os.getcwd())
try:
    import torch
    print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu         ', torch.cuda.get_device_name(0))
except ImportError:
    print('torch        not installed yet')
for mod in ('google.colab.userdata', 'google.colab.drive', 'google.colab.files'):
    try:
        __import__(mod)
        print(f'{mod:24s} available')
    except Exception as exc:
        print(f'{mod:24s} NOT available ({type(exc).__name__})')

python       3.12.13
cwd          /content/hebrew-negation-embeddings
torch        2.11.0+cu128 | cuda: True
gpu          Tesla T4
google.colab.userdata    available
google.colab.drive       available
google.colab.files       available


**Prints** &nbsp; `GH_TOKEN:` and where it came from, pip's log, then the last 3
commits — the top one should be the newest `nli:` commit.

**Writes** &nbsp; The repo at `/content/hebrew-negation-embeddings` on the VM. The
working directory moves into it, so every path after this is relative to the repo root.

In [6]:
OWNER, REPO, BRANCH = 'ItayBoros', 'hebrew-negation-embeddings', 'main'

gh_token = get_secret('GH_TOKEN')
url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'
bare_url = f'https://github.com/{OWNER}/{REPO}.git'

if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,url,REPO], check=True)
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else '.')

# Re-authenticate origin before every fetch, not just on first clone: the repo is
# private, and the last step below strips the token, so a second run in the same
# session would otherwise fetch unauthenticated and fail.
subprocess.run(['git','remote','set-url','origin', url], check=True)
subprocess.run(['git','fetch','-q','origin',BRANCH], check=True)

# reset --hard, not pull: this VM's checkout is scratch space that always mirrors
# origin, never its own source of truth (every notebook here commits from your
# machine, not from the VM), so local drift must never block picking up a new
# push. `pull` (a merge) refuses whenever an untracked file would be overwritten,
# and scripts in this project routinely create exactly that - writing straight
# into results/ before those paths exist in git - so a later sync hits "untracked
# working tree files would be overwritten by merge" the moment that path is
# committed for real.
subprocess.run(['git','reset','-q','--hard', f'origin/{BRANCH}'], check=True)

# drop the token from the stored remote so it is not left on the VM's disk
subprocess.run(['git','remote','set-url','origin', bare_url], check=True)
del gh_token, url

!pip install -q -r requirements.txt
!git log --oneline -3

db14511 (HEAD -> main, origin/main, origin/HEAD) nli: name the LCHAIM checkpoint's results like the fine-tuned model's
0797abb nli: fetch + reset --hard instead of pull, so a stray result file can't block sync
6e3aa1a nli: point nli_rerank at our checkpoint, and compare against the released one


**Prints** &nbsp; `all pipeline checks passed`, `all NLI data checks passed`,
`all NLI eval checks passed`, `all run_eval checks passed`. Anything else: stop here.

**Writes** &nbsp; Nothing.

In [7]:
# offline checks first - seconds, no network, no GPU
!python -m tests.test_data_pipeline | tail -3
!python -m tests.test_nli_data | tail -3
!python -m tests.test_nli_eval | tail -3
!python -m tests.test_run_eval | tail -3

wrote 2 new/updated rows -> /tmp/tmpsvk9lnll/r.csv (2 rows total)

all pipeline checks passed
== results table keeps one row per configuration ==

all NLI data checks passed
== check_test_file: the row-count guard ==

all NLI eval checks passed
== append_or_replace: baseline/projection rows (blank nli_* fields) survive too ==

all run_eval checks passed


## 2. Data — the clean HebNLI test split

Same file `03_eval_nli.ipynb` used: `data/raw/hebnli_test_clean.jsonl`, 883 rows,
built by dropping the 689 held-out promptIDs and auditing for text overlap with the
probe. Gitignored, so a fresh VM has to rebuild it — this checks first and only
redoes the work if it is missing or the wrong size.

**Prints** &nbsp; Whether the file is already here with the right count.

**Writes** &nbsp; Nothing.

In [8]:
from pathlib import Path

TEST_CLEAN = Path('data/raw/hebnli_test_clean.jsonl')
n_existing = sum(1 for _ in TEST_CLEAN.open(encoding='utf-8')) if TEST_CLEAN.exists() else 0

NEEDS_REGEN = n_existing != 883
print(f'{TEST_CLEAN}: {n_existing} rows found' if n_existing else f'{TEST_CLEAN}: not found')
print('regeneration needed:', NEEDS_REGEN)

data/raw/hebnli_test_clean.jsonl: 883 rows found
regeneration needed: False


**Prints** &nbsp; `HF_TOKEN:` and where it came from — skipped if not needed.

**Writes** &nbsp; Nothing. The token stays in memory.

In [9]:
if NEEDS_REGEN:
    os.environ['HF_TOKEN'] = get_secret('HF_TOKEN')
else:
    print('skipped - clean test file already present with the right row count')

skipped - clean test file already present with the right row count


**Prints** &nbsp; Skipped if not needed. Otherwise the same funnel as
`03_eval_nli.ipynb`: `rows 884` then `kept 883`.

**Writes** &nbsp; `data/raw/hebnli_test.jsonl` and `data/raw/hebnli_test_clean.jsonl` on
the VM, gitignored.<br>`results/nli_data_test.json` — the manifest, committed.

In [10]:
if NEEDS_REGEN:
    !python -m src.data.hebnli --split test --out data/raw/hebnli_test.jsonl
    !python -m src.nli.prepare_data --source data/raw/hebnli_test.jsonl --split test --out data/raw/hebnli_test_clean.jsonl
else:
    print('skipped - nothing to regenerate')

skipped - nothing to regenerate


**Prints** &nbsp; The funnel, then the row count. Must read exactly `883`.

**Writes** &nbsp; Nothing. It only reads the manifest and the file back.

In [11]:
import json

manifest = json.load(open('results/nli_data_test.json', encoding='utf-8'))
f = manifest['funnel']
print(f"loaded={f['loaded']}  id_filter=-{f['loaded']-f['prompt_id_clean']}"
      f"  text_audit=-{manifest['text_overlap']['rows_dropped']}  kept={manifest['rows_written']}")

n_rows = sum(1 for _ in TEST_CLEAN.open(encoding='utf-8'))
assert n_rows == 883, f'expected exactly 883 rows in {TEST_CLEAN}, found {n_rows}'
assert manifest['rows_written'] == 883
print(f'\n{TEST_CLEAN}: {n_rows} rows confirmed')

loaded=884  id_filter=-1  text_audit=-0  kept=883

data/raw/hebnli_test_clean.jsonl: 883 rows confirmed


## 3. Our checkpoint on Drive

**Prints** &nbsp; Drive's mount confirmation and a check that all four expected
files are there. No fallback: our checkpoint only exists on Drive.

**Writes** &nbsp; Nothing beyond mounting Drive at `/content/drive`.

In [12]:
CKPT = '/content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    raise RuntimeError(
        'Drive did not mount. Our checkpoint only exists on Drive - make sure this '
        f'session is signed in with the same Google account used for training. '
        f'({type(exc).__name__}: {exc})'
    ) from exc

ckpt_dir = Path(CKPT)
if not ckpt_dir.is_dir():
    raise RuntimeError(f'{CKPT} not found on Drive - check this mounted the right account')
for name in ('model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json'):
    if not (ckpt_dir / name).exists():
        raise RuntimeError(f'{name} missing from {CKPT}')
print('checkpoint dir:', CKPT)
print('found: model.safetensors, config.json, tokenizer.json, tokenizer_config.json')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoint dir: /content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean
found: model.safetensors, config.json, tokenizer.json, tokenizer_config.json


## 4. Part A — HebNLI test-set accuracy: ours vs. the released checkpoint

Both against the same 883-row clean test file, so the numbers are directly
comparable in form — accuracy, macro-F1, per-class, confusion matrix. Whether the
*comparison* itself is fair is a separate question, addressed by the caveat printed
below the released checkpoint's numbers.

**Prints** &nbsp; Skipped with a note if `results/nli_test_alephbert-hebnli-clean.json`
already exists (from `03_eval_nli.ipynb`) — no need to burn GPU time twice for a
number that cannot change. Otherwise the same output `03_eval_nli.ipynb` produced.

**Writes** &nbsp; `results/nli_test_alephbert-hebnli-clean.json` and
`results/nli_test_predictions.csv`, only if not already present.

In [13]:
OUR_SUMMARY = 'results/nli_test_alephbert-hebnli-clean.json'
if Path(OUR_SUMMARY).exists():
    print(f'skipped - {OUR_SUMMARY} already exists (from 03_eval_nli.ipynb)')
else:
    !python -m src.nli.eval_nli --checkpoint {CKPT} \
        --test data/raw/hebnli_test_clean.jsonl --expected-n 883 \
        --summary-out {OUR_SUMMARY} \
        --predictions-out results/nli_test_predictions.csv

skipped - results/nli_test_alephbert-hebnli-clean.json already exists (from 03_eval_nli.ipynb)


**Prints** &nbsp; `label source assumed(released)`, `pair encoding joined`, then
accuracy / macro-F1 / per-class / confusion matrix, then the `[caveat]` line — read
that line before repeating this number anywhere.

**Writes** &nbsp; `results/nli_test_alephbert-hebnli-lchaim.json` and
`results/nli_test_alephbert-hebnli-lchaim_predictions.csv`, a few hundred KB, on the
VM — a separate pair of files from ours, named the same way.

In [14]:
!python -m src.nli.eval_nli --checkpoint oriel9p/AlephBERT-FT-HebNLI-LCHAIM \
    --test data/raw/hebnli_test_clean.jsonl --expected-n 883 \
    --summary-out results/nli_test_alephbert-hebnli-lchaim.json \
    --predictions-out results/nli_test_alephbert-hebnli-lchaim_predictions.csv

config.json: 100% 926/926 [00:00<00:00, 4.32MB/s]
tokenizer_config.json: 100% 1.44k/1.44k [00:00<00:00, 953kB/s]
vocab.txt: 100% 545k/545k [00:00<00:00, 9.80MB/s]
tokenizer.json: 100% 1.37M/1.37M [00:00<00:00, 17.8MB/s]
special_tokens_map.json: 100% 695/695 [00:00<00:00, 4.71MB/s]

AlephBERT/model.safetensors: downloading bytes:  44% 221M/504M [00:02<00:01, 213MB/s, 18.0MB/s  ]  
AlephBERT/model.safetensors: reconstructing file:  27% 134M/504M [00:03<00:08, 44.3MB/s]
AlephBERT/model.safetensors: downloading bytes:  92% 465M/504M [00:04<00:00, 198MB/s, 37.1MB/s  ]  ]
AlephBERT/model.safetensors: reconstructing file:  80% 402M/504M [00:04<00:01, 89.1MB/s, 23.1MB/s  ]
AlephBERT/model.safetensors: downloading bytes: 100% 479M/479M [00:08<00:00, 58.0MB/s, 38.0MB/s  ] ]
AlephBERT/model.safetensors: reconstructing file: 100% 504M/504M [00:08<00:00, 61.0MB/s, 33.3MB/s  ]
Loading weights: 100% 201/201 [00:00<00:00, 18570.31it/s]
checkpoint           oriel9p/AlephBERT-FT-HebNLI-LCHAIM
test file 

**Prints** &nbsp; Both summaries' headline numbers side by side.

**Writes** &nbsp; Nothing. It only reads the two summary files back.

In [15]:
import json

ours = json.load(open('results/nli_test_alephbert-hebnli-clean.json', encoding='utf-8'))
lchaim = json.load(open('results/nli_test_alephbert-hebnli-lchaim.json', encoding='utf-8'))

print(f"{'metric':16s} {'ours (clean)':>14s} {'lchaim':>14s}")
for key in ('accuracy', 'macro_precision', 'macro_recall', 'macro_f1'):
    print(f'{key:16s} {ours[key]:>14.4f} {lchaim[key]:>14.4f}')

if lchaim.get('caveat'):
    print(f"\n[caveat on 'lchaim'] {lchaim['caveat']}")

metric             ours (clean)         lchaim
accuracy                 0.7961         0.7271
macro_precision          0.7946         0.7386
macro_recall             0.7943         0.7217
macro_f1                 0.7940         0.7220

[caveat on 'lchaim'] this checkpoint was fine-tuned on all of HebNLI rather than a train/test split, so this test set was very likely part of its own training data - treat this score as a reference point, not a fair comparison


## 5. Part B — the negation-probe experiment: nli_rerank, both checkpoints

`nli_rerank.py` blends embedding cosine with `P(entailment) - P(contradiction)`:

    score = (1 - lam) * cosine + lam * nli_score

`lam` defaults to 1.0 (pure NLI), which means the embedder argument does not affect
`nli_rerank`'s score at all under the default — `cosine` is computed but never used
in the blend. So unlike the `baseline`/`projection` sweep, there is no reason to run
`nli_rerank` against all four frozen embedders; one is run below (`multilingual-e5`,
the strongest baseline performer) purely so the row has a `model` value, not because
the choice changes the number.

Two runs into the same `--out`: the released checkpoint (the current default), then
ours. `append_or_replace` (in `run_eval.py`) keys rows on
`(model, intervention, nli_checkpoint, nli_encoding, nli_lam)`, so the second run
adds a row beside the first instead of erasing it.

**Prints** &nbsp; The checkpoint path, `encoding joined` (or `pair`), the label
names, six pairs each `[ok]`/`[MISMATCH]`, and a tally — confirms the label mapping
before either harness run below trusts it.

**Writes** &nbsp; Nothing.

In [16]:
print('=== released checkpoint ===')
!python -m src.interventions.check_nli_labels

print('\n=== our checkpoint ===')
!python -m src.interventions.check_nli_labels --model {CKPT} --subfolder ""

=== released checkpoint ===
model     oriel9p/AlephBERT-FT-HebNLI-LCHAIM
encoding  joined
Loading weights: 100% 201/201 [00:00<00:00, 5499.27it/s]
labels    {0: 'contradiction', 1: 'entailment', 2: 'neutral'}

Expected: entailment
 contradiction: 0.107616
    entailment: 0.711746
       neutral: 0.180637
Predicted: entailment  [ok]

Expected: contradiction
 contradiction: 0.925731
    entailment: 0.026635
       neutral: 0.047634
Predicted: contradiction  [ok]

Expected: neutral
 contradiction: 0.356381
    entailment: 0.154891
       neutral: 0.488728
Predicted: neutral  [ok]

Expected: contradiction
 contradiction: 0.633170
    entailment: 0.150569
       neutral: 0.216261
Predicted: contradiction  [ok]

Expected: contradiction
 contradiction: 0.714755
    entailment: 0.089335
       neutral: 0.195910
Predicted: contradiction  [ok]

Expected: entailment
 contradiction: 0.302515
    entailment: 0.413263
       neutral: 0.284222
Predicted: entailment  [ok]

6/6 agree with the assumed l

**Prints** &nbsp; `[ok] multilingual-e5 nli_rerank gap=... nevir=...`, then
`wrote 1 new/updated rows -> results/results_nli_rerank.csv (1 rows total)`.

**Writes** &nbsp; `results/results_nli_rerank.csv` — one row, the released checkpoint.

In [17]:
!python -m src.harness.run_eval \
    --probe data/probe/probe.jsonl \
    --models multilingual-e5 --interventions nli_rerank \
    --out results/results_nli_rerank.csv

modules.json: 100% 387/387 [00:00<00:00, 1.68MB/s]
README.md: 100% 179k/179k [00:00<00:00, 96.5MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 266kB/s]
config.json: 100% 694/694 [00:00<00:00, 3.36MB/s]

model.safetensors: downloading bytes:  14% 155M/1.11G [00:01<00:04, 219MB/s, 11.9MB/s  ]  
model.safetensors: downloading bytes:  17% 193M/1.11G [00:01<00:05, 168MB/s, 17.1MB/s  ]
model.safetensors: reconstructing file:  18% 201M/1.11G [00:01<00:06, 150MB/s, 6.51MB/s  ]
model.safetensors: downloading bytes:  34% 378M/1.11G [00:02<00:02, 249MB/s, 32.2MB/s  ] ]
model.safetensors: downloading bytes:  67% 741M/1.11G [00:03<00:01, 245MB/s, 61.8MB/s  ] ]
model.safetensors: reconstructing file:  54% 604M/1.11G [00:06<00:06, 78.6MB/s, 30.4MB/s  ]
model.safetensors: downloading bytes: 100% 741M/741M [00:08<00:00, 90.1MB/s, 61.6MB/s  ]  ]
model.safetensors: reconstructing file: 100% 1.11G/1.11G [00:08<00:00, 135MB/s, 80.9MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 22913.40it

**Prints** &nbsp; The same shape, `(2 rows total)` this time.

**Writes** &nbsp; A second row in `results/results_nli_rerank.csv`, beside the first
rather than replacing it.

In [18]:
!python -m src.harness.run_eval \
    --probe data/probe/probe.jsonl \
    --models multilingual-e5 --interventions nli_rerank \
    --nli-model {CKPT} --nli-subfolder "" --nli-encoding pair \
    --out results/results_nli_rerank.csv

Loading weights: 100% 199/199 [00:00<00:00, 6494.04it/s]
Loading weights: 100% 201/201 [00:00<00:00, 3342.62it/s]
[ok] multilingual-e5  nli_rerank   gap=+1.822 nevir=0.99

wrote 1 new/updated rows -> results/results_nli_rerank.csv (2 rows total)


**Prints** &nbsp; The two-row comparison table: `nli_checkpoint`,
`cosine_gap`, `nevir_rank` side by side for the released checkpoint and ours.

**Writes** &nbsp; Nothing. It only reads the csv back.

In [19]:
import pandas as pd

df = pd.read_csv('results/results_nli_rerank.csv')
df[['nli_checkpoint', 'nli_encoding', 'cosine_gap', 'nevir_rank']]

,nli_checkpoint,nli_encoding,cosine_gap,nevir_rank
0,oriel9p/AlephBERT-FT-HebNLI-LCHAIM,joined,1.141438,0.986755
1,/content/drive/MyDrive/hebrew-negation/checkpo...,pair,1.822266,0.993377


## 6. Download the results

**Prints** &nbsp; Browser downloads, or the files printed inline when that is
unavailable — printed unconditionally this time (see `03_eval_nli.ipynb`'s section 7:
under the VS Code extension `files.download()` can return without raising while the
file lands nowhere findable, so the inline print is not gated on an exception).

**Writes** &nbsp; Your machine, as downloads and/or as text in the saved notebook.

In [20]:
RESULTS = [
    'results/nli_test_alephbert-hebnli-lchaim.json',
    'results/nli_test_alephbert-hebnli-lchaim_predictions.csv',
    'results/results_nli_rerank.csv',
]
try:
    from google.colab import files
    for path in RESULTS:
        files.download(path)
except Exception as exc:
    print(f'[warn] browser download unavailable ({type(exc).__name__})')

for path in RESULTS:
    print(f'\n===== {path} =====')
    print(open(path, encoding='utf-8').read())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


===== results/nli_test_alephbert-hebnli-lchaim.json =====
{
  "checkpoint": "oriel9p/AlephBERT-FT-HebNLI-LCHAIM",
  "test_file": "data/raw/hebnli_test_clean.jsonl",
  "n_examples": 883,
  "label_ids": {
    "entailment": 0,
    "neutral": 1,
    "contradiction": 2
  },
  "label_source": "assumed(released)",
  "pair_encoding": "joined",
  "caveat": "this checkpoint was fine-tuned on all of HebNLI rather than a train/test split, so this test set was very likely part of its own training data - treat this score as a reference point, not a fair comparison",
  "accuracy": 0.7270668176670442,
  "macro_precision": 0.7386171816466721,
  "macro_recall": 0.7216527931205262,
  "macro_f1": 0.7219591901801371,
  "per_class": {
    "entailment": {
      "precision": 0.75,
      "recall": 0.7722772277227723,
      "f1": 0.7609756097560976,
      "support": 303
    },
    "neutral": {
      "precision": 0.7929292929292929,
      "recall": 0.575091575091575,
      "f1": 0.6666666666666666,
      "suppo

Commit the three result files from your machine with the `nli:` prefix.

`results/nli_test_alephbert-hebnli-clean.json` / `results/nli_test_predictions.csv`
are unchanged from `03_eval_nli.ipynb` if this run skipped re-computing them — nothing
new to commit there unless section 4's first cell actually ran.